# Intro to Plotly

Plotly is a library for building **interactive** charts: hover for values, zoom, pan, toggle legend entries, and export to HTML.

It has two main interfaces:

| Interface | Import | When to use |
|---|---|---|
| **Plotly Express** | `import plotly.express as px` | High-level, one-line charts from tidy DataFrames (start here) |
| **Graph Objects** | `import plotly.graph_objects as go` | Low-level, full control, building figures piece by piece |

Every Plotly chart, whichever interface made it, is a **`Figure`** object with two parts:
- `data`: a list of **traces** (each trace is one set of marks: a scatter series, a bar series, ...)
- `layout`: everything that isn't data (title, axes, legend, size, template, ...)

Install it with `pip install plotly` (and `pip install nbformat` if figures don't render in Jupyter).

---
## Section 1: Plotly Express basics and the Figure object

### 1.1 Your first figure
Plotly Express functions take a DataFrame plus **column names** for each visual channel (`x`, `y`, `color`, `size`, ...). They return a `Figure`; call `.show()` (or just leave `fig` as the last line of a cell) to display it.

Plotly ships sample datasets under `px.data`. We'll use `iris`, `tips`, and `gapminder` a lot.

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

iris = px.data.iris()
iris.head()

In [ ]:
fig = px.scatter(iris, x="sepal_width", y="sepal_length")
fig.show()

### 1.2 Mapping columns to visual channels
The power of Express is that one argument maps a column onto a visual property, and Plotly builds the legend for you.

| Argument | Effect |
|---|---|
| `color` | colour by column (categorical → discrete colours, numeric → colour scale) |
| `size` | marker size by a numeric column |
| `symbol` | marker shape by a categorical column |
| `hover_name` | bold title in the hover tooltip |
| `hover_data` | extra columns shown on hover (list, or dict to show/hide/format) |
| `title` | figure title |
| `labels` | dict renaming axis/legend labels, e.g. `{"sepal_width": "Sepal width (cm)"}` |

In [ ]:
fig = px.scatter(
    iris,
    x="sepal_width", y="sepal_length",
    color="species",
    size="petal_length",
    hover_data=["petal_width"],
    title="Iris: sepal dimensions",
    labels={"sepal_width": "Sepal width (cm)", "sepal_length": "Sepal length (cm)"},
)
fig.show()

### 1.3 Line charts
`px.line` works the same way. Data should be sorted by `x`, and `color` (or `line_group`) splits it into one line per group. `markers=True` adds points.

In [ ]:
gap = px.data.gapminder()
oceania = gap[gap["continent"] == "Oceania"]
fig = px.line(oceania, x="year", y="lifeExp", color="country", markers=True)
fig.show()

### 1.4 Looking inside a Figure
A figure is just a nested, dict-like structure. Inspecting it is the best way to understand Plotly.

In [ ]:
fig = px.scatter(iris, x="sepal_width", y="sepal_length", color="species")

print(type(fig))
print("Number of traces:", len(fig.data))   # one trace per species!
print(fig.data[0].name, fig.data[0].type, fig.data[0].mode)
print(fig.layout.xaxis.title.text)

Notice that `color="species"` produced **three traces**, one per category. That's how Express builds legends: each legend entry is a trace.

### 1.5 Updating a figure after creation
Rather than cramming everything into the Express call, you can modify the figure afterwards:

- `fig.update_layout(...)`: title, size, legend, fonts, template...
- `fig.update_traces(...)`: marker/line properties for **all** traces (or a subset via `selector=dict(...)`)
- `fig.update_xaxes(...)` / `fig.update_yaxes(...)`: axis range, type, gridlines...

Nested properties can be written with **magic underscores**: `title_font_size=20` is the same as `title=dict(font=dict(size=20))`.

In [ ]:
fig = px.scatter(iris, x="sepal_width", y="sepal_length", color="species")
fig.update_layout(title="Updated afterwards", width=700, height=450, legend_title_text="Species")
fig.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=1, color="black")))
fig.update_traces(marker_symbol="diamond", selector=dict(name="setosa"))  # only one trace
fig.update_xaxes(range=[1.5, 5])
fig.show()

### 1.6 Saving figures
- `fig.write_html("chart.html")`: a standalone interactive file (no extra dependencies)
- `fig.write_image("chart.png")`: static image (requires `pip install kaleido`)

---
## Section 1 Problems

Write your code in the empty cell below each problem. Use the built-in datasets.

**Problem 1.1**: Load `px.data.tips()` into a variable `tips`. Make a scatter plot of `total_bill` (x) against `tip` (y).

**Problem 1.2**: Redo the tips scatter, colouring by `sex` and using `symbol` for `smoker`. Give it the title "Tips vs total bill". How many traces does the resulting figure have? Print `len(fig.data)` and explain the number in a comment.

**Problem 1.3**: Make a tips scatter where marker size reflects `size` (party size), hovering shows `day` as the bold hover name and also shows `time`. Rename axes to "Total bill ($)" and "Tip ($)" using `labels`.

**Problem 1.4**: Using `gapminder`, plot a line chart of `gdpPercap` over `year` for the countries `["China", "India", "United States", "Brazil"]`, one line per country, with markers. Hint: `df["country"].isin([...])`.

**Problem 1.5**: Take your figure from 1.4 and, **without** re-calling `px.line`, put the y-axis on a log scale (`type="log"`), set the figure height to 500, and move the legend title to read "Country".

**Problem 1.6**: Create the iris scatter coloured by species. Then use `update_traces` with a `selector` so that only the `virginica` trace has marker size 14, while the others stay default. Finally, loop over `fig.data` and print each trace's `name` and `marker.size`.

**Problem 1.7**: Without using Plotly's sample data, build your own DataFrame: `x` from 0 to 10 (100 points), and two columns `sin` and `cos`. Plot both as lines on the same chart with **one** `px.line` call. Hint: `y` can take a list of columns, *or* you can `melt` the DataFrame into long format and use `color`. Try both if you can.

**Problem 1.8 (challenge)**: Print `fig.to_dict()["layout"]` for any figure you've made, then find the key holding the template. Next, re-render the figure with `fig.update_layout(template="plotly_dark")`, then with `"simple_white"`. Save the one you prefer as `my_first_plotly.html`.